In [4]:
import numpy as np
import pandas as pd
import os
import glob
import pandas as pd
import re

from synchronization_script import *

In [5]:
def process_trial(trial_path, video_dir):
    subject_id = trial_path.split("/")[7].split("_")[1]
    trial_id = trial_path.split("/")[7].split("_")[2]

    print(f"Subject ID: {subject_id}")
    print(f"Trial ID: {trial_id}")

    output_dir = f"/standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed/{subject_id}_{trial_id}/synched_data/"
    os.makedirs(output_dir, exist_ok=True)
    output_csv_path = os.path.join(output_dir, f"final_annotation_{subject_id}_{trial_id}.csv")


    # # find the video file in specified directory
    # video_files = glob.glob(os.path.join(video_dir, "*.mp4"))
    # if not video_files:
    #     print(f"No video file found in {video_dir}")
    #     return
    # video_file = video_files[0]  # assuming there's only one video file
    # print(f"Video file found: {video_file}")

    # # copy this file to output directory with a new name
    # output_video_path = os.path.join(output_dir, f"{subject_id}_{trial_id}.mp4")
    # os.system(f"cp '{video_file}' '{output_video_path}'")
    # print(f"Copied video to {output_video_path}")


    # read in the final annotation CSV
    df = pd.read_csv(trial_path)

    trakstar_df = df[[col for col in df.columns if 'trakstar' in col]]

    print("Trakstar Columns:")
    print(trakstar_df.columns.tolist())

    # transform the trakstar data
    transformed_trakstar_df = build_trakstar_positions_in_raven(trakstar_df)  


    print("Transformed Trakstar DataFrame Head:")
    print(transformed_trakstar_df.columns.tolist())

    # # plot trakstar positions to verify correctness side by side before and after transformation
    # import matplotlib.pyplot as plt
    # fig, axs = plt.subplots(2, 3, figsize=(15, 10))
    # axs[0, 0].plot(trakstar_df['trakstar_sensor_1_x'], label='Original X')
    # axs[0, 0].set_title('Original Trakstar Sensor 1 X Position')
    # axs[0, 1].plot(trakstar_df['trakstar_sensor_1_y'], label='Original Y')
    # axs[0, 1].set_title('Original Trakstar Sensor 1 Y Position')
    # axs[0, 2].plot(trakstar_df['trakstar_sensor_1_z'], label='Original Z')
    # axs[0, 2].set_title('Original Trakstar Sensor 1 Z Position')
    # axs[1, 0].plot(transformed_trakstar_df['trakstar_sensor_1_x_transformed'], label='Transformed X', color='orange')
    # axs[1, 0].set_title('Transformed Trakstar Sensor 1 X Position')
    # axs[1, 1].plot(transformed_trakstar_df['trakstar_sensor_1_y_transformed'], label='Transformed Y', color='orange')
    # axs[1, 1].set_title('Transformed Trakstar Sensor 1 Y Position')
    # axs[1, 2].plot(transformed_trakstar_df['trakstar_sensor_1_z_transformed'], label='Transformed Z', color='orange')
    # axs[1, 2].set_title('Transformed Trakstar Sensor 1 Z Position')
    # for ax in axs.flat:
    #     ax.legend()
    # plt.tight_layout()
    # plt.show()

    # add/update trakstar columns from transformed_trakstar_df into df (align by index)
    # this will overwrite existing columns and create any new columns present in transformed_trakstar_df
    df.loc[:, transformed_trakstar_df.columns] = transformed_trakstar_df

    print("Final DataFrame Columns:")
    print(df.columns.tolist())

    # drop columns if all values are -1
    cols_to_drop = [col for col in df.columns if (df[col] == -1).all()]
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped columns with all -1 values: {cols_to_drop}")

    # drop specified unnecessary columns
    unnecessary_columns = ["server_time", "trakstar_time_ns", "trakstar_delta_ns"]
    df.drop(columns=unnecessary_columns, inplace=True)

    # save the updated dataframe to a new CSV
    df.to_csv(output_csv_path, index=False)
    print(f"Saved processed data to {output_csv_path}")


In [6]:
trial_dir = "/standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/"

# iterate through the immediate subdirectories in trial_dir
for subject_dir in os.listdir(trial_dir):
    subject_path = os.path.join(trial_dir, subject_dir)
    if os.path.isdir(subject_path):
        trial_path = os.path.join(subject_path, "synched_data", "final_annotation_remapped.csv")
        video_dir = os.path.join(subject_path, "video")
        if os.path.isfile(trial_path):
            print(f"Processing file: {trial_path}")
            process_trial(trial_path, video_dir)

Processing file: /standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/VentralHR2_S217_T4_2024-07-19/synched_data/final_annotation_remapped.csv
Subject ID: S217
Trial ID: T4
Trakstar Columns:
['trakstar_time_ns', 'trakstar_delta_ns', 'trakstar_sensor_0_azimuth', 'trakstar_sensor_1_azimuth', 'trakstar_sensor_2_azimuth', 'trakstar_sensor_3_azimuth', 'trakstar_sensor_0_elevation', 'trakstar_sensor_1_elevation', 'trakstar_sensor_2_elevation', 'trakstar_sensor_3_elevation', 'trakstar_sensor_0_roll', 'trakstar_sensor_1_roll', 'trakstar_sensor_2_roll', 'trakstar_sensor_3_roll', 'trakstar_sensor_0_x', 'trakstar_sensor_1_x', 'trakstar_sensor_2_x', 'trakstar_sensor_3_x', 'trakstar_sensor_0_y', 'trakstar_sensor_1_y', 'trakstar_sensor_2_y', 'trakstar_sensor_3_y', 'trakstar_sensor_0_z', 'trakstar_sensor_1_z', 'trakstar_sensor_2_z', 'trakstar_sensor_3_z']
Building TrakStar positions...
Transformed Trakstar DataFrame Head:
['trakstar_time_ns', 'trakstar_delta_ns', 'trakstar_sensor_0_azimuth', 'trakst

/tmp/ipykernel_13042/3259227756.py:28: DtypeWarning: Columns (70,71,72,73) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(trial_path)


Saved processed data to /standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed/S214_T4/synched_data/final_annotation_S214_T4.csv
Processing file: /standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/VENTRAL_S118_T1_2024-07-19/synched_data/final_annotation_remapped.csv
Subject ID: S118
Trial ID: T1


/tmp/ipykernel_13042/3259227756.py:28: DtypeWarning: Columns (70,71,72,73,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(trial_path)


Trakstar Columns:
['trakstar_time_ns', 'trakstar_delta_ns', 'trakstar_sensor_0_azimuth', 'trakstar_sensor_1_azimuth', 'trakstar_sensor_2_azimuth', 'trakstar_sensor_3_azimuth', 'trakstar_sensor_0_elevation', 'trakstar_sensor_1_elevation', 'trakstar_sensor_2_elevation', 'trakstar_sensor_3_elevation', 'trakstar_sensor_0_roll', 'trakstar_sensor_1_roll', 'trakstar_sensor_2_roll', 'trakstar_sensor_3_roll', 'trakstar_sensor_0_x', 'trakstar_sensor_1_x', 'trakstar_sensor_2_x', 'trakstar_sensor_3_x', 'trakstar_sensor_0_y', 'trakstar_sensor_1_y', 'trakstar_sensor_2_y', 'trakstar_sensor_3_y', 'trakstar_sensor_0_z', 'trakstar_sensor_1_z', 'trakstar_sensor_2_z', 'trakstar_sensor_3_z']
Building TrakStar positions...
Transformed Trakstar DataFrame Head:
['trakstar_time_ns', 'trakstar_delta_ns', 'trakstar_sensor_0_azimuth', 'trakstar_sensor_1_azimuth', 'trakstar_sensor_2_azimuth', 'trakstar_sensor_3_azimuth', 'trakstar_sensor_0_elevation', 'trakstar_sensor_1_elevation', 'trakstar_sensor_2_elevation', '

...

Trakstar Columns:
['trakstar_time_ns', 'trakstar_delta_ns', 'trakstar_sensor_0_azimuth', 'trakstar_sensor_1_azimuth', 'trakstar_sensor_2_azimuth', 'trakstar_sensor_3_azimuth', 'trakstar_sensor_0_elevation', 'trakstar_sensor_1_elevation', 'trakstar_sensor_2_elevation', 'trakstar_sensor_3_elevation', 'trakstar_sensor_0_roll', 'trakstar_sensor_1_roll', 'trakstar_sensor_2_roll', 'trakstar_sensor_3_roll', 'trakstar_sensor_0_x', 'trakstar_sensor_1_x', 'trakstar_sensor_2_x', 'trakstar_sensor_3_x', 'trakstar_sensor_0_y', 'trakstar_sensor_1_y', 'trakstar_sensor_2_y', 'trakstar_sensor_3_y', 'trakstar_sensor_0_z', 'trakstar_sensor_1_z', 'trakstar_sensor_2_z', 'trakstar_sensor_3_z']
Building TrakStar positions...
Transformed Trakstar DataFrame Head:
['trakstar_time_ns', 'trakstar_delta_ns', 'trakstar_sensor_0_azimuth', 'trakstar_sensor_1_azimuth', 'trakstar_sensor_2_azimuth', 'trakstar_sensor_3_azimuth', 'trakstar_sensor_0_elevation', 'trakstar_sensor_1_elevation', 'trakstar_sensor_2_elevation', '

/tmp/ipykernel_13042/3259227756.py:28: DtypeWarning: Columns (70,71,72,73,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(trial_path)


Trakstar Columns:
['trakstar_time_ns', 'trakstar_delta_ns', 'trakstar_sensor_0_azimuth', 'trakstar_sensor_1_azimuth', 'trakstar_sensor_2_azimuth', 'trakstar_sensor_3_azimuth', 'trakstar_sensor_0_elevation', 'trakstar_sensor_1_elevation', 'trakstar_sensor_2_elevation', 'trakstar_sensor_3_elevation', 'trakstar_sensor_0_roll', 'trakstar_sensor_1_roll', 'trakstar_sensor_2_roll', 'trakstar_sensor_3_roll', 'trakstar_sensor_0_x', 'trakstar_sensor_1_x', 'trakstar_sensor_2_x', 'trakstar_sensor_3_x', 'trakstar_sensor_0_y', 'trakstar_sensor_1_y', 'trakstar_sensor_2_y', 'trakstar_sensor_3_y', 'trakstar_sensor_0_z', 'trakstar_sensor_1_z', 'trakstar_sensor_2_z', 'trakstar_sensor_3_z']
Building TrakStar positions...
Transformed Trakstar DataFrame Head:
['trakstar_time_ns', 'trakstar_delta_ns', 'trakstar_sensor_0_azimuth', 'trakstar_sensor_1_azimuth', 'trakstar_sensor_2_azimuth', 'trakstar_sensor_3_azimuth', 'trakstar_sensor_0_elevation', 'trakstar_sensor_1_elevation', 'trakstar_sensor_2_elevation', '

/tmp/ipykernel_13042/3259227756.py:28: DtypeWarning: Columns (70,71,72,73,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(trial_path)


Dropped columns with all -1 values: ['Pedal 3 Pressure']
Saved processed data to /standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed/S202_T1/synched_data/final_annotation_S202_T1.csv
